# IMDb Source Profiling

This notebook inspects the raw IMDb review dataset before ingestion. It checks the structure of individual reviews, counts records across the six source files, and profiles field availability and missing values.

## 1. Set up libraries and source path

In [1]:
from pathlib import Path
from collections import Counter

import ijson
import pandas as pd

# Folder where the original IMDb review files are stored
source_folder = Path(r"C:\Data\imdb_reviews")

# Sample file used to inspect the review structure
sample_file = source_folder / "sample.json"

# The six files that make up the full review dataset
part_files = sorted(source_folder.glob("part-*.json"))

## 2. Inspect sample reviews

Read a small number of reviews from the sample file to understand the source structure without loading the full dataset into memory.

In [2]:
sample_reviews = []

with open(sample_file, "rb") as f:
    reviews = ijson.items(f, "item")

    for i, review in enumerate(reviews):
        sample_reviews.append(review)

        if i == 9:
            break

df_sample = pd.DataFrame(sample_reviews)

df_sample

,review_id,reviewer,movie,rating,review_summary,review_date,spoiler_tag,review_detail,helpful
0,rw1133942,OriginalMovieBuff21,Kill Bill: Vol. 2 (2004),8,Good follow up that answers all the questions,24 July 2005,0,"After seeing Tarantino's Kill Bill Vol: 1, I g...","[0, 1]"
1,rw1133943,sentra14,Journey to the Unknown (1968– ),NaN,Excellent series,24 July 2005,0,"I have the entire series on video, taped mostl...","[11, 11]"
2,rw1133946,GreenwheelFan2002,The Island (2005),9,"Not just about action, but about survival...",24 July 2005,0,Once again the critics prove themselves as mor...,"[2, 5]"
3,rw1133948,itsascreambaby,Win a Date with Tad Hamilton! (2004),3,Falls under the category: seen it a million ti...,24 July 2005,0,This IS a film that has been done too many tim...,"[2, 3]"
4,rw1133949,OriginalMovieBuff21,Saturday Night Live: The Best of Chris Farley ...,10,"Before Tommy Boy and Black Sheep, there was Sa...",24 July 2005,0,Chris Farley is one of my favorite comedians a...,"[4, 4]"
5,rw1133950,Aaron1375,Outlaw Star (1998– ),10,Great anime series soars through the stars.,24 July 2005,0,"I love this anime series, my only complaint is...","[11, 12]"
6,rw1133952,TheFilmConnoisseur,The Aviator (2004),10,Howard Hughes for Dummies,24 July 2005,0,****Excellent ***Good **Fair *Poor Before watc...,"[0, 2]"
7,rw1133953,swansongang,Star Wars: Episode I - The Phantom Menace (1999),9,Better than people say,24 July 2005,1,I always get annoyed when people say how bad t...,"[7, 10]"
8,rw1133954,diand_,The Amityville Horror (2005),3,Laid-back horror,24 July 2005,0,The Amityville Horror is once again a horror m...,"[0, 1]"
9,rw1133955,btillman63,Flying Tigers (1942),6,Tigers Opted Out,24 July 2005,0,Several friends of mine flew with the AVG. One...,"[19, 29]"


## 3. Profile the full source dataset

Scan all six source files once to count the reviews, confirm the fields are consistent, and check for missing values.

In [3]:
from collections import Counter

# Fields expected in each IMDb review
expected_fields = [
    "review_id",
    "reviewer",
    "movie",
    "rating",
    "review_summary",
    "review_date",
    "spoiler_tag",
    "review_detail",
    "helpful"
]

total_reviews = 0
field_counts = Counter()
missing_counts = Counter()

# Go through each source file without loading the full dataset into memory
for file in part_files:
    file_count = 0

    with open(file, "rb") as f:
        for review in ijson.items(f, "item"):
            total_reviews += 1
            file_count += 1

            # Count how often each field appears in the source data
            for field in review.keys():
                field_counts[field] += 1

            # Count fields that are missing or do not contain a value
            for field in expected_fields:
                if field not in review or review[field] is None or review[field] == "":
                    missing_counts[field] += 1

    print(f"Finished {file.name}: {file_count:,} reviews")

print("\nTOTAL:", f"{total_reviews:,}")

print("\nFIELD COUNTS")
for field in expected_fields:
    print(f"{field}: {field_counts[field]:,}")

print("\nMISSING VALUES")
for field in expected_fields:
    print(f"{field}: {missing_counts[field]:,}")

Finished part-01.json: 1,010,293 reviews
Finished part-02.json: 1,012,212 reviews
Finished part-03.json: 1,015,000 reviews
Finished part-04.json: 1,019,000 reviews
Finished part-05.json: 1,014,997 reviews
Finished part-06.json: 499,997 reviews

TOTAL: 5,571,499

FIELD COUNTS
review_id: 5,571,499
reviewer: 5,571,499
movie: 5,571,499
rating: 5,571,499
review_summary: 5,571,499
review_date: 5,571,499
spoiler_tag: 5,571,499
review_detail: 5,571,499
helpful: 5,571,499

MISSING VALUES
review_id: 0
reviewer: 0
movie: 0
rating: 662,849
review_summary: 13
review_date: 0
spoiler_tag: 0
review_detail: 1
helpful: 0


## 4. Profiling summary

The IMDb source contains 5,571,499 reviews across six JSON files, with nine fields consistently present across all records.

Missing data is concentrated mainly in `rating`, with 662,849 missing values. There are also 13 missing `review_summary` values and 1 missing `review_detail`. The remaining fields have no missing values.

These results provide the source baseline used to validate the IMDb ingestion process.